In [11]:
import pandas as pd
import numpy as np

In [12]:
# Chargement du dataset (ajuste le chemin si nécessaire)
df = pd.read_csv(r'C:\Users\PC\Documents\Projet_Chlordecone_2026\data\BaseCLD2026.csv', sep=';', encoding='UTF-8')
# Afficher les 5 premières lignes
print(df.head())


      ID  ANNEE   COMMU_LAB       RAIN Sol_simple  \
0  20143   2010  GROS-MORNE  2000-3000    Andosol   
1  20143   2010  GROS-MORNE  2000-3000    Andosol   
2  20143   2010  GROS-MORNE  2000-3000    Andosol   
3  20143   2010  GROS-MORNE  2000-3000    Andosol   
4  20143   2010  GROS-MORNE  2000-3000    Andosol   

                                            type_sol Date_prelevement  \
0  Intergrades Sols … allophane relativement ‚vol...       24/05/2007   
1  Intergrades Sols … allophane relativement ‚vol...       24/05/2007   
2  Intergrades Sols … allophane relativement ‚vol...       24/05/2007   
3  Intergrades Sols … allophane relativement ‚vol...       24/05/2007   
4  Intergrades Sols … allophane relativement ‚vol...       24/05/2007   

  Date_enregistrement Date_analyse Operateur_chld  ...  Taux_5b_hydro  \
0          24/05/2007   24/05/2007              =  ...           0,07   
1          24/05/2007   24/05/2007              =  ...           0,07   
2          24/05/2007  

In [13]:
# Voir les types de colonnes et les valeurs non-nulles
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 31126 entries, 0 to 31125
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     31126 non-null  int64  
 1   ANNEE                  31126 non-null  int64  
 2   COMMU_LAB              30828 non-null  str    
 3   RAIN                   31126 non-null  str    
 4   Sol_simple             31052 non-null  str    
 5   type_sol               28517 non-null  str    
 6   Date_prelevement       31126 non-null  str    
 7   Date_enregistrement    31126 non-null  str    
 8   Date_analyse           31126 non-null  str    
 9   Operateur_chld         31126 non-null  str    
 10  Taux_Chlordecone       31126 non-null  float64
 11  Operateur_5b           31126 non-null  str    
 12  Taux_5b_hydro          31114 non-null  str    
 13  histoBanane_Histo_ban  13143 non-null  float64
 14  mnt_tpi_mean           31098 non-null  float64
 15  mnt_tri_mean 

Convertion des colonnes dates

In [15]:
# Liste des colonnes de dates
colonnes_dates = ['Date_prelevement', 'Date_enregistrement', 'Date_analyse']

# Conversion en type datetime
for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Vérification du nouveau type
print(df[colonnes_dates].dtypes)
print(df.info())

Date_prelevement       datetime64[us]
Date_enregistrement    datetime64[us]
Date_analyse           datetime64[us]
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 31126 entries, 0 to 31125
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   ID                     31126 non-null  int64         
 1   ANNEE                  31126 non-null  int64         
 2   COMMU_LAB              30828 non-null  str           
 3   RAIN                   31126 non-null  str           
 4   Sol_simple             31052 non-null  str           
 5   type_sol               28517 non-null  str           
 6   Date_prelevement       31126 non-null  datetime64[us]
 7   Date_enregistrement    31126 non-null  datetime64[us]
 8   Date_analyse           31126 non-null  datetime64[us]
 9   Operateur_chld         31126 non-null  str           
 10  Taux_Chlordecone       31126 non-null  float64       
 11  Op

detection d'incohérence sur les valeur de taux_5b_hydro 

In [16]:
# On crée un masque pour isoler les lignes où la valeur contient des lettres
masque_texte = df['Taux_5b_hydro'].astype(str).str.contains('[a-zA-Z]', na=False)

# On affiche ces lignes spécifiques
incoherences = df[masque_texte]

print(f"Nombre de lignes incohérentes : {len(incoherences)}")
print(incoherences[['ID', 'COMMU_LAB', 'Taux_5b_hydro']].drop_duplicates())


print(f"Nombre total de NaN après nettoyage : {df['Taux_5b_hydro'].isna().sum()}")

Nombre de lignes incohérentes : 597
          ID     COMMU_LAB Taux_5b_hydro
7      20462         DUCOS           inf
119    20315   MARIGOT(LE)           inf
180    20326   MARIGOT(LE)           inf
332    20150   VAUCLIN(LE)           inf
461    20172   LORRAIN(LE)           inf
...      ...           ...           ...
19898  20215  SAINT-JOSEPH           inf
20473  20192  SAINT-ESPRIT           inf
20475  20230  BASSE-POINTE           inf
20696  20187    GROS-MORNE       d�tect�
20792  20349  BASSE-POINTE           inf

[94 rows x 3 columns]
Nombre total de NaN après nettoyage : 12


On observe la présence de la mention 'détecté' ou 'inf 'au lieu d'une valeur numérique. Pour la suite de l'analyse statistique, nous avons choisi de transformer ces mentions en NaN pour ne pas fausser les moyennes, tout en conservant une trace de ces prélèvements.


In [17]:

# Harmonisation des virgules (indispensable pour les chiffres normaux)
df['Taux_5b_hydro'] = df['Taux_5b_hydro'].astype(str).str.replace(',', '.')

# Remplacement ciblé de "détecté" par NaN
# On utilise une regex pour attraper toutes les variantes (détecté, dtect, etc.)
df['Taux_5b_hydro'] = df['Taux_5b_hydro'].replace(to_replace=r'.*tect.*', value=np.nan, regex=True)

# Conversion en numérique SANS toucher aux "inf"
# On ne peut pas utiliser pd.to_numeric(errors='coerce') ici car il transformerait les 'inf' en NaN.
# On va donc convertir ligne par ligne ce qui ressemble à un nombre.
def conversion_selective(valeur):
    if str(valeur).lower() == 'inf':
        return np.inf
    try:
        return float(valeur)
    except:
        return np.nan

df['Taux_5b_hydro'] = df['Taux_5b_hydro'].apply(conversion_selective)

print("Vérification :")
print(f"- Nombre de 'inf' préservés : {np.isinf(df['Taux_5b_hydro']).sum()}")
print(f"- Nombre de 'NaN' (anciens 'détecté') : {df['Taux_5b_hydro'].isna().sum()}")

Vérification :
- Nombre de 'inf' préservés : 586
- Nombre de 'NaN' (anciens 'détecté') : 23


Traitement des autres colonnesnumériques de la meme façon

In [18]:
# Sélection des colonnes numériques
cols_numeriques = df.select_dtypes(include=['float64', 'int64']).columns

# Affichage du résumé
#print(df[cols_numeriques].describe())
print(df[cols_numeriques].isna().sum())

ID                           0
ANNEE                        0
Taux_Chlordecone             0
Taux_5b_hydro               23
histoBanane_Histo_ban    17983
mnt_tpi_mean                28
mnt_tri_mean                28
mnt_rugosite_mean           28
mnt_ombrage_mean            28
mnt_exposition_mean         28
mnt_pente_mean              28
X                            0
Y                            0
dtype: int64


Pour traiter les valeurs manquantes :
Plutôt qu'une imputation par la médiane globale, ou la moyenne, j'ai opté pour un algorithme k-NN. Ce choix est plus cohérent car il respecte la continuité spatiale du relief : les caractéristiques topographiques d'une parcelle sont estimées à partir de ses voisins géographiques les plus proches.

Pour cela il va faloir traiter d'abord les 'inf'

In [20]:
print("Colonnes contenant des valeurs infinies :")
print(np.isinf(df[cols_a_imputer]).sum())

Colonnes contenant des valeurs infinies :
X                        0
Y                        0
mnt_tpi_mean             0
mnt_tri_mean             0
mnt_rugosite_mean        0
mnt_ombrage_mean         0
mnt_exposition_mean      0
mnt_pente_mean           0
Taux_5b_hydro          586
dtype: int64


Remplacement des valeurs 'inf' par le maximum des taux_5b_hydro multiplié par 1.1

In [23]:
#Identifier la valeur maximale RÉELLE en ignorant les infinis
valeur_max = df.loc[~np.isinf(df['Taux_5b_hydro']), 'Taux_5b_hydro'].max()

# Créer la valeur "Saturée" (Max + 10%)
valeur_saturee = valeur_max * 1.1

# Remplacer les 'inf' par cette valeur 
df['Taux_5b_hydro'] = df['Taux_5b_hydro'].replace([np.inf, -np.inf], valeur_saturee)

print("Colonnes contenant des valeurs infinies :")
print(np.isinf(df[cols_a_imputer]).sum())

Colonnes contenant des valeurs infinies :
X                      0
Y                      0
mnt_tpi_mean           0
mnt_tri_mean           0
mnt_rugosite_mean      0
mnt_ombrage_mean       0
mnt_exposition_mean    0
mnt_pente_mean         0
Taux_5b_hydro          0
dtype: int64


En ce moment on peut executer l'algorithme k-nn pour remplacer les valeurs Naan

In [24]:
from sklearn.impute import KNNImputer

# 1. On s'assure qu'on ne travaille que sur les colonnes numériques
cols_a_imputer = ['X', 'Y', 'mnt_tpi_mean', 'mnt_tri_mean', 
                  'mnt_rugosite_mean', 'mnt_ombrage_mean', 
                  'mnt_exposition_mean', 'mnt_pente_mean', 'Taux_5b_hydro']

# 2. Initialisation (5 voisins est un excellent compromis)
imputer = KNNImputer(n_neighbors=5)

# 3. Lancement de l'imputation
# On utilise .values pour accélérer le calcul interne de scikit-learn
df[cols_a_imputer] = imputer.fit_transform(df[cols_a_imputer])

# 4. LE TEST FINAL
print("--- Vérification du nettoyage (Volet 4.6) ---")
print("Les colonnes ayant des Naan après application du K-nn")
print(df[cols_a_imputer].isna().sum())

--- Vérification du nettoyage (Volet 4.6) ---
Les colonnes ayant des Naan après application du K-nn
X                      0
Y                      0
mnt_tpi_mean           0
mnt_tri_mean           0
mnt_rugosite_mean      0
mnt_ombrage_mean       0
mnt_exposition_mean    0
mnt_pente_mean         0
Taux_5b_hydro          0
dtype: int64


Détection de valeurs abérantes

In [25]:

# Liste de tes colonnes numériques
cols_analyse = ['mnt_tpi_mean', 'mnt_tri_mean', 'mnt_rugosite_mean', 
                 'mnt_ombrage_mean', 'mnt_exposition_mean', 'mnt_pente_mean', 'Taux_5b_hydro']

def synthese_outliers(df, colonnes):
    resultats = []
    
    for col in colonnes:
        # Calcul des quartiles
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Calcul des seuils (Méthode 1.5 * IQR)
        seuil_bas = Q1 - 1.5 * IQR
        seuil_haut = Q3 + 1.5 * IQR
        
        # Identification des lignes concernées
        nb_outliers_bas = len(df[df[col] < seuil_bas])
        nb_outliers_haut = len(df[df[col] > seuil_haut])
        total_outliers = nb_outliers_bas + nb_outliers_haut
        pourcentage = (total_outliers / len(df)) * 100
        
        resultats.append({
            'Colonne': col,
            'Seuil Bas': round(seuil_bas, 2),
            'Seuil Haut': round(seuil_haut, 2),
            'Nb Outliers': total_outliers,
            '% Outliers': round(pourcentage, 2)
        })
    
    return pd.DataFrame(resultats)

# Affichage du bilan
bilan_outliers = synthese_outliers(df, cols_analyse)
print("--- SYNTHÈSE DES VALEURS ATYPIQUES ---")
print(bilan_outliers)

--- SYNTHÈSE DES VALEURS ATYPIQUES ---
               Colonne  Seuil Bas  Seuil Haut  Nb Outliers  % Outliers
0         mnt_tpi_mean      -4.09        4.48         2304        7.40
1         mnt_tri_mean      -2.50       10.50          909        2.92
2    mnt_rugosite_mean      -7.91       33.18          801        2.57
3     mnt_ombrage_mean     108.41      244.21          698        2.24
4  mnt_exposition_mean    -149.47      488.46            0        0.00
5       mnt_pente_mean     -13.64       51.55          822        2.64
6        Taux_5b_hydro      -0.02        0.04         4410       14.17


In [26]:
df[df['mnt_pente_mean'] < 0]

,ID,ANNEE,COMMU_LAB,RAIN,Sol_simple,type_sol,Date_prelevement,Date_enregistrement,Date_analyse,Operateur_chld,...,Taux_5b_hydro,histoBanane_Histo_ban,mnt_tpi_mean,mnt_tri_mean,mnt_rugosite_mean,mnt_ombrage_mean,mnt_exposition_mean,mnt_pente_mean,X,Y


Le colonnes numériques sont bien traités en ce moment
Interressons nous aux traitement des colonnes textes